# SST DIO: Minian parameter review (G27 HC2)

Plan: `plans/sst_dio_dreadd_validation_plan.md` §6. Each grid variant changes **one** value from the frozen set; every variant has the same lens-interior seed mask. Kernel: `minian-native` (Minian's own viewers).

1. the summary table; 2. any variant against `frozen`, side by side (saved as a PNG for Claude); 3. Minian's `CNMFViewer` on one variant; 4. log each decision.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.expanduser('~/code/caban_sst'))
sys.path.insert(0, os.path.expanduser('~/code/minian_vsekulic'))

import holoviews as hv
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
from minian.visualization import CNMFViewer, VArrayViewer

from caban import sst_dio_parameter_review as review

%matplotlib inline
hv.notebook_extension('bokeh', width=100)
output_size = 100
GRID_ROOT = os.path.expanduser('~/sst_dio_out/parameter_grid/G27_HC2')
review.summary_table(GRID_ROOT)

## 2. A variant against `frozen`

Left: frozen (orange outlines). Middle: the variant (blue). Right: both outlines over the variant's max projection. Pick from the list.

In [ ]:
finished = [name for name, r in review.variants(GRID_ROOT).items()
            if r['status'] == 'done' and name != 'frozen']


@widgets.interact(variant=widgets.Dropdown(options=finished))
def show(variant):
    review.compare(GRID_ROOT, variant)
    plt.show()

## 3. Minian's unit viewer on one variant

Set `VARIANT`, run the cell. Footprints, traces (C, S) and the motion-corrected movie, as in the pipeline notebook's CNMF viewer.

In [ ]:
VARIANT = 'frozen'

A = review.open_array(GRID_ROOT, VARIANT, 'A')
C = review.open_array(GRID_ROOT, VARIANT, 'C')
S = review.open_array(GRID_ROOT, VARIANT, 'S')
Y_fm_chk = review.open_array(GRID_ROOT, VARIANT, 'Y_fm_chk', intermediate=True)
cnmfviewer = CNMFViewer(A=A, C=C, S=S, org=Y_fm_chk)
hv.output(size=int(output_size * 0.7))
display(cnmfviewer.show())

## 4. Log a decision

One line per stage decided; it goes to `parameter_log.csv` beside the grid, which Claude reads.

In [ ]:
# review.log_decision(GRID_ROOT, stage='background removal (tophat wnd)',
#                     values_tried='10, 15, 20', value_chosen='15', why='...')